# Model Iteration Notebook 

In this notebook, we will focus on training and evaluating multiple machine learning models to classify emotions in transcribed television show data. 

The different secitons in this notebook will help demonstrate how different models perform on the same dataset and will provide insights into model selection and optimization. 

For every type of model and/or iteration that we did, we tracked the details in the model iteration file that you can find attached to the final assignment on Brightspace or here [Model Iteration File](link-here). In order to improve our models, we used (a selection of) traditional NLP features that we extracted in the ‘NLP Features’ task [NLP Features.ipynb notebook](NLP Features.ipynb). 



In [46]:
import sys
import os

# Get the directory of the current notebook (which is /notebooks/)
# and navigate up one level (to /project_root/) to find /src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the system path
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can import as if you were in the project root
# For /src/processing/features.py, the module is src.processing.features
from src.processing.features import FeatureEngine

[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [47]:
import numpy as np
import pandas as pd 
from datasets import DatasetDict, load_dataset

In [48]:
import torch 

print("Torch version:",torch.__version__)

print("Is CUDA enabled?",torch.cuda.is_available())

Torch version: 2.5.1
Is CUDA enabled? True


## Loading the datasets

### Dataset 1 - Sentiment and Emotion Analysis Dataset

The dataset can be found at [https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download](https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download)

The dataset contains over 422,000 sentences, labeled with six distinct emotions:

- Joy: 143,067 samples
- Sadness: 121,187 samples
- Anger: 59,317 samples
- Fear: 49,649 samples
- Love: 34,554 samples
- Surprise: 14,972 samples


In [49]:
# https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download
# Sentiment and Emotion Analysis Dataset

df1 = pd.read_csv(r"..\Data\CSV\sentiment_data\combined_emotion.csv")
df1['emotion'].value_counts()

emotion
joy        143067
sad        121187
anger       59317
fear        49649
love        34554
suprise     14972
Name: count, dtype: int64

### Dataset 2 - roskoN/dailydialog

The dataset can be found at [https://huggingface.co/datasets/roskoN/dailydialog](https://huggingface.co/datasets/roskoN/dailydialog)

The dataset contains 102,979 sentences, labeled with seven distinct emotions, including neutral:

- Neutral: 85572
- Happiness: 12885
- Surprise: 1823
- Hadness: 1150
- Anger: 1022
- Disgust: 353
- Fear: 174


In [50]:
ds = load_dataset("roskoN/dailydialog")

## Preprocessing

### Dataset 1 

1. Emotion Harmonization: The emotion `"love"` was removed because it did not align with the six primary emotions used in the final schema.

2. Name Standardization: The emotion names `"joy"` and `"sad"` were renamed to `"happiness"` and `"sadness"`, respectively, to match the target naming convention.  

3. Typo Correction: A typo was corrected by changing the misspelled emotion `"suprise"` to the correct spelling, `"surprise"`, resolving mapping issues. 

4. Column Renaming: The column names `sentence` and `emotion` were capitalized to `Sentence` and `Emotion` to match the consistent naming used for Dataset 2.

In [51]:
values_to_remove = ['love']

mask_to_keep = ~df1['emotion'].isin(values_to_remove) # ~ Means reverse condition
df1 = df1[mask_to_keep]

In [52]:
df1.replace('joy', 'happiness', inplace=True) # Replace joy with happiness to match the naming convention
df1.replace('sad', 'sadness', inplace=True) # Replace sad with sadness to match the naming convention
df1.replace('suprise', 'surprise', inplace=True) # Fix typo suprise --> surprise

In [53]:
df1.rename(columns={
    'sentence': 'Sentence',
    'emotion': 'Emotion'
}, inplace=True)

### Dataset 2 

Here's a slightly rephrased and condensed explanation of the steps taken:

To prepare the data:

1. Emotion Mapping: The specific numerical IDs were obtained by accessing the original research paper linked on Hugging Face. The paper can be found at [https://aclanthology.org/I17-1099/](https://aclanthology.org/I17-1099/). I then downloaded the provided zip file (`I17-1099.Datasets.zip`), and inside it, the file `readme.txt` contained the ID-to-emotion mapping we used.

2. Reverse Map Creation: A secondary, inverted dictionary was created to map the emotion names back to their corresponding IDs. This was done to ensure consistency and standardize the emotion IDs across multiple datasets (e.g., mapping emotion names in Dataset 1 to match the ID scheme of this dataset).

3. Dataframe Conversion & Unpacking: The Hugging Face DatasetDict object was converted into a pandas DataFrame. During this process, rows originally containing a list of multiple sentences and their corresponding emotions were unpacked to create a single row for every individual sentence-emotion pair.

In [54]:
# Emotion Map obtained from https://aclanthology.org/I17-1099/ --> Download I17-1099.Datasets.zip --> readme.txt 
emotion_map = {0: "neutral", 
              1: "anger", 
              2: "disgust", 
              3: "fear", 
              4: "happiness", 
              5: "sadness", 
              6: "surprise"} 

# Rever Emotion Map to match the IDs of dataset 1 with the ones of dataset 2 
reverse_map  = {v: k for k, v in emotion_map.items()}

In [55]:
def make_dataset_into_df(dataset: DatasetDict) -> pd.DataFrame:
    """
    Flattens a Hugging Face DatasetDict into a single pandas DataFrame
    where each row is a sentence-emotion pair.
    """
    all_sentences = []
    all_emotions = []
    all_splits = []

    # Iterate through each split ('train', 'validation', 'test')
    for split_name, ds_split in dataset.items():
        # Iterate through each dialogue/example in the split
        for example in ds_split:
            # 'utterances' and 'emotions' are lists
            for sentence, emotion in zip(example['utterances'], example['emotions']):
                all_sentences.append(sentence)
                all_emotions.append(emotion)
                all_splits.append(split_name) # Keep track of which split it came from

    # Create the DataFrame from the collected lists
    df = pd.DataFrame({
        'Sentence': all_sentences,
        'Emotion_ID': all_emotions,
        'Split': all_splits  
    })

    return df

In [56]:
df2 = make_dataset_into_df(ds)        

In [57]:
df2['Emotion'] = df2['Emotion_ID'].map(emotion_map)

In [58]:
df2['Emotion'].value_counts()

Emotion
neutral      85572
happiness    12885
surprise      1823
sadness       1150
anger         1022
disgust        353
fear           174
Name: count, dtype: int64

## Merging the Datasets

In [59]:
df_concat = pd.concat([df1, df2], ignore_index=True)

In [60]:
df_concat['Emotion_ID'] = df_concat['Emotion_ID'].fillna(df_concat['Emotion'].map(reverse_map))
df_concat['Emotion_ID'].astype(int)
df_concat['Emotion'].value_counts()

Emotion
happiness    155952
sadness      122337
neutral       85572
anger         60339
fear          49823
surprise      16795
disgust         353
Name: count, dtype: int64

### Data preparation for scikit-learn

In [71]:
scikit_data = []
# Get unique emotions
unique_emotions = df_concat['Emotion'].unique()

# For each unique emotion, add 350 different rows
for emotion in unique_emotions:
    # Get all rows with this emotion
    emotion_rows = df_concat[df_concat['Emotion'] == emotion]
    
    # Sample 350 rows (with replacement if there are fewer than 350 available)
    sampled_rows = emotion_rows.sample(n=350, replace=True, random_state=42)
    
    # Add to the list
    scikit_data.append(sampled_rows)

# Create new dataframe by concatenating all sampled rows
scikit_data = pd.concat(scikit_data, ignore_index=True)

scikit_data.drop(columns=['Split'], inplace=True)
scikit_data['Emotion_ID'] = scikit_data['Emotion_ID'].astype(int)

scikit_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2450 entries, 0 to 2449
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence    2450 non-null   object
 1   Emotion     2450 non-null   object
 2   Emotion_ID  2450 non-null   int32 
dtypes: int32(1), object(2)
memory usage: 48.0+ KB


In [73]:
scikit_data['Sentence'].nunique()
scikit_data.drop_duplicates(subset=['Sentence'], inplace=True)

In [74]:
features = FeatureEngine(transcript_input=scikit_data, output_path=None)

results = features.create_features()

Training Word2Vec model...
Training complete.


In [122]:
scikit_df = results.copy()

In [123]:
results.head()

,Emotion_ID,Sentiment,word2vec_embedding,bert_embedding
0,3,-0.187500,"[0.026285807, 0.05158488, 0.038472492, 0.10248...","[0.23701519, 0.1843421, -0.0013413723, -0.3083..."
1,3,0.142857,"[-0.014211019, -0.019111633, 0.052668255, 0.09...","[0.104994155, 0.3022204, 0.16726117, 0.0569860..."
2,3,0.000000,"[0.022728443, -0.041015625, 0.033935547, 0.037...","[-0.3623942, -0.19997108, -0.5202047, 0.025784..."
3,3,-0.208333,"[0.025583902, 0.048528034, -0.041046143, 0.059...","[-0.2753944, 0.20885071, -0.41002783, -0.34576..."
4,3,0.125000,"[0.0017972905, 0.0356577, 0.023579698, 0.07220...","[-0.20424555, 0.19182691, -0.31577486, -0.3621..."


In [124]:
scikit_df = scikit_df.drop(columns=['Emotion', 'POS_tags','TF-IDF','custom_word2vec_embedding', 'Sentence'])

KeyError: "['Emotion', 'POS_tags', 'TF-IDF', 'custom_word2vec_embedding', 'Sentence'] not found in axis"

In [125]:
scikit_df['Sentiment'] = scikit_df['Sentiment'].values.reshape(-1, 1)
scikit_df['word2vec_embedding'] = np.vstack(scikit_df['word2vec_embedding'].values)
scikit_df['bert_embedding'] = np.vstack(scikit_df['bert_embedding'].values)

### Logistic Regression

In [126]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

log_reg = LogisticRegression(max_iter=1000, random_state=42)
X = scikit_df.drop(columns=['Emotion_ID'])
y = scikit_df['Emotion_ID']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

print(classification_report(y_test, y_pred, target_names=emotion_map.values()))


              precision    recall  f1-score   support

     neutral       0.32      0.40      0.36        73
       anger       0.22      0.33      0.27        70
     disgust       0.36      0.10      0.15        41
        fear       0.21      0.26      0.23        69
   happiness       0.29      0.71      0.41        55
     sadness       0.24      0.07      0.11        80
    surprise       0.20      0.03      0.05        72

    accuracy                           0.26       460
   macro avg       0.26      0.27      0.23       460
weighted avg       0.26      0.26      0.22       460



In [128]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB(var_smoothing=0.0005)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print(classification_report(y_test, y_pred_nb, target_names=emotion_map.values()))

              precision    recall  f1-score   support

     neutral       0.31      0.37      0.34        73
       anger       0.14      0.11      0.13        70
     disgust       0.41      0.17      0.24        41
        fear       0.25      0.28      0.26        69
   happiness       0.25      0.73      0.37        55
     sadness       0.37      0.28      0.31        80
    surprise       0.00      0.00      0.00        72

    accuracy                           0.27       460
   macro avg       0.25      0.28      0.24       460
weighted avg       0.24      0.27      0.23       460



In [129]:
from sklearn.svm import LinearSVC

svc = LinearSVC(penalty='l2', loss='squared_hinge', C=500.0, multi_class='ovr', fit_intercept=True, random_state=42)
svc.fit(X_train, y_train)

y_pred_svc = svc.predict(X_test)
print(classification_report(y_test, y_pred_svc, target_names=emotion_map.values()))

              precision    recall  f1-score   support

     neutral       0.28      0.47      0.35        73
       anger       0.24      0.30      0.27        70
     disgust       0.67      0.05      0.09        41
        fear       0.27      0.22      0.24        69
   happiness       0.24      0.76      0.37        55
     sadness       0.21      0.05      0.08        80
    surprise       0.00      0.00      0.00        72

    accuracy                           0.26       460
   macro avg       0.27      0.26      0.20       460
weighted avg       0.25      0.26      0.20       460



c:\Users\filip\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\filip\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\filip\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result